# CityFlow OSNet-AIN Strong Baseline — Colab Training Notebook

**8-hour Colab plan:** OSNet-AIN-x1.0 + ID loss + triplet loss + bag-of-tricks + offline re-ranking.

| Section | What it does |
|---|---|
| **1. Runtime setup** | GPU check, Drive mount, torchreid-from-source install |
| **2. Configuration** | All knobs in one dict |
| **3. Dataset registration** | Paste your existing `cityflow_aic21` wrapper here |
| **4. Datamanager** | Build with random-erase, color-jitter, RandomIdentitySampler |
| **5. Model / optimizer / scheduler** | OSNet-AIN, BNNeck, last_stride=1, cosine LR + warmup |
| **6. Train** | 120 epochs, eval every 20, checkpoints to Drive |
| **7. Evaluation** | Without re-ranking (live path) + with k-reciprocal (offline upper-bound) |
| **8. Results table** | Fill in new numbers, exports Markdown + CSV |
| **9. Feature export** | Save embeddings for later offline re-ranking |
| **10. Figures + artefacts** | CMC curve, bar ablation, checkpoint copy |
| **11. Threshold-sweep CSV** | `eval_finetuned_osnet_v4.csv` → Drive |
| **12. Practical notes** | Resume tips, memory, what to skip |

**Before running:** Runtime → Change runtime type → GPU (T4 / A100)

## 1. Runtime setup

In [1]:
!nvidia-smi

Fri May 29 09:40:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             57W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%%capture
!pip install -q Cython
!git clone https://github.com/KaiyangZhou/deep-person-reid.git
%cd /content/deep-person-reid
!pip install -q -r requirements.txt
!python setup.py develop -q

In [4]:
import os, gc, random, time
from pathlib import Path
import numpy as np
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab: Runtime -> Change runtime type -> GPU.')

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
DEVICE = 'cuda'
print('GPU  :', torch.cuda.get_device_name(0))
print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')

GPU  : NVIDIA A100-SXM4-80GB
VRAM : 85.09 GB


## 2. Configuration

In [5]:
CFG = {
    # ── Paths ──────────────────────────────────────────────────────────────
    'data_root':    '/content/drive/MyDrive/cityflow',
    'save_dir':     '/content/drive/MyDrive/cityflow/output/logs',
    'thesis_dir':   '/content/drive/MyDrive/Thesis',

    'train_csv':    '/content/drive/MyDrive/Thesis/osnet_train.csv',
    'val_split':    0.15,


    # ── Model ──────────────────────────────────────────────────────────────
    'model_name':   'osnet_ain_x1_0',
    'dataset_name': 'cityflow',

    # ── Input ──────────────────────────────────────────────────────────────
    'height': 256,
    'width':  256,

    # ── Batch / sampling ───────────────────────────────────────────────────
    'batch_size_train': 128,   # P=16, K=4
    'batch_size_test':  128,
    'num_instances':    4,
    'workers':          2,    # T4 Colab: keep ≤2

    # ── Optimiser ──────────────────────────────────────────────────────────
    'optim':        'adam',
    'lr':           3.5e-4,
    'weight_decay': 5e-4,

    # ── Schedule ───────────────────────────────────────────────────────────
    'max_epoch':     80,
    'warmup_epochs': 10,

    # ── Loss ───────────────────────────────────────────────────────────────
    'margin':   0.3,
    'weight_t': 1.0,   # triplet loss weight
    'weight_x': 1.0,   # ID (cross-entropy) loss weight

    # ── Evaluation ─────────────────────────────────────────────────────────
    'eval_freq':   10,
    'print_freq':  50,
    'dist_metric': 'euclidean',

    # ── Time-budget fallback ───────────────────────────────────────────────
    # Set True to run 80 epochs instead of 120 (saves ~1.5 h on a free T4)
    'use_80_epoch_fallback': True,
}

if CFG['use_80_epoch_fallback']:
    CFG['max_epoch']     = 80
    CFG['warmup_epochs'] = 8

for d in (CFG['save_dir'], CFG['thesis_dir']):
    Path(d).mkdir(parents=True, exist_ok=True)

print('Config ready.')
print(f"  Model     : {CFG['model_name']}  256×256")
print(f"  Epochs    : {CFG['max_epoch']}  (warmup {CFG['warmup_epochs']})")
print(f"  Batch     : {CFG['batch_size_train']}  (num_instances={CFG['num_instances']})")
print(f"  Save dir  : {CFG['save_dir']}")

Config ready.
  Model     : osnet_ain_x1_0  256×256
  Epochs    : 80  (warmup 8)
  Batch     : 128  (num_instances=4)
  Save dir  : /content/drive/MyDrive/cityflow/output/logs


In [6]:
# # ── 2b. Extract ZIPs to local SSD (fast I/O during training) ─────────────────
# # Copies ZIPs from Drive to /content/ (SSD) once, then extracts resumably.
# # Re-run after any disconnect — the marker file prevents double-extraction.

# import shutil, zipfile, os
# from pathlib import Path
# from tqdm.notebook import tqdm

# DRIVE_REAL_ZIP = '/content/drive/MyDrive/cityflow/AICity21_Track2_ReID.zip'
# DRIVE_SIM_ZIP  = '/content/drive/MyDrive/cityflow/AICity21_Track2_ReID_Simulation.zip'

# LOCAL_REAL_ZIP = '/content/AICity21_Track2_ReID.zip'
# LOCAL_SIM_ZIP  = '/content/AICity21_Track2_ReID_Simulation.zip'

# LOCAL_DATA     = '/content/cityflow_data'     # real images land here
# LOCAL_SIM_DATA = '/content/sim_data'          # synthetic images land here
# EXTRACT_MARKER = '.extract_complete'          # written when extraction finishes

# # ── helpers ───────────────────────────────────────────────────────────────────
# def ensure_local_zip(drive_path, local_path, label):
#     if os.path.exists(local_path):
#         print(f'{label} already on SSD ({os.path.getsize(local_path)/1e9:.2f} GB). Skipping copy.')
#         return
#     print(f'Copying {label} from Drive to SSD …')
#     shutil.copy(drive_path, local_path)
#     print(f'{label} done ({os.path.getsize(local_path)/1e9:.2f} GB)')

# def extract_split_resumable(zip_path, dst_dir, split_keyword, label=None):
#     """Extract images whose zip-internal path contains split_keyword into dst_dir.
#     Skips entirely if marker file exists (resumable after disconnect)."""
#     dst_dir = Path(dst_dir)
#     marker  = dst_dir / EXTRACT_MARKER
#     lbl     = label or dst_dir.name

#     if marker.exists():
#         n = sum(1 for _ in dst_dir.rglob('*.jpg'))
#         print(f'{lbl} already extracted ({n} images). Skipping.')
#         return

#     dst_dir.mkdir(parents=True, exist_ok=True)
#     with zipfile.ZipFile(zip_path, 'r') as zf:
#         members = [m for m in zf.namelist()
#                    if f'/{split_keyword}/' in m and m.endswith('.jpg')]
#         for m in tqdm(members, desc=lbl, unit='file', leave=False):
#             zf.extract(m, dst_dir)
#     marker.touch()
#     n = sum(1 for _ in dst_dir.rglob('*.jpg'))
#     print(f'{lbl}: {n} images extracted.')

# def extract_xml_from_zip(zip_path, dst_dir):
#     """Extract all XML files from a ZIP flat into dst_dir."""
#     dst_dir = Path(dst_dir)
#     dst_dir.mkdir(parents=True, exist_ok=True)
#     with zipfile.ZipFile(zip_path, 'r') as zf:
#         for m in [x for x in zf.namelist() if x.endswith('.xml')]:
#             fname = Path(m).name
#             (dst_dir / fname).write_bytes(zf.read(m))
#             print(f'  Extracted XML: {fname}')

# # ── real CityFlowV2 ───────────────────────────────────────────────────────────
# print('=== Real data (CityFlowV2) ===')
# ensure_local_zip(DRIVE_REAL_ZIP, LOCAL_REAL_ZIP, 'Real ZIP')
# for split in ('image_train', 'image_test', 'image_query'):
#     extract_split_resumable(LOCAL_REAL_ZIP, f'{LOCAL_DATA}/{split}', split, split)

# # ── synthetic VehicleX ────────────────────────────────────────────────────────
# print('\n=== Synthetic data (VehicleX) ===')
# ensure_local_zip(DRIVE_SIM_ZIP, LOCAL_SIM_ZIP, 'Sim ZIP')
# extract_split_resumable(LOCAL_SIM_ZIP, f'{LOCAL_SIM_DATA}/sys/image_train',
#                         'sys/image_train', 'synthetic images')
# extract_xml_from_zip(LOCAL_SIM_ZIP, LOCAL_SIM_DATA)

# # ── update CFG so downstream cells use the SSD paths ─────────────────────────
# CFG['local_data']     = LOCAL_DATA
# CFG['local_sim_data'] = LOCAL_SIM_DATA
# print('\n✓ All data ready on local SSD.')
# print(f"  Real images  : {LOCAL_DATA}")
# print(f"  Sim images   : {LOCAL_SIM_DATA}")

In [7]:
import os
import zipfile
import shutil
from tqdm.notebook import tqdm

REAL_ZIP = '/content/drive/MyDrive/cityflow/AICity21_Track2_ReID.zip'
REAL_DEST = '/content/cityflow_data'

print('=== FORCE extracting SIM ZIP ===\n')

# Delete old destination completely
if os.path.exists(REAL_DEST):
    print(f'Deleting {REAL_DEST}...')
    shutil.rmtree(REAL_DEST)

os.makedirs(REAL_DEST, exist_ok=True)

# Extract everything
with zipfile.ZipFile(REAL_ZIP) as zf:
    all_files = zf.namelist()
    jpgs = [f for f in all_files if f.endswith('.jpg')]

    print(f'ZIP contents: {len(all_files)} total, {len(jpgs):,} JPGs\n')
    print('Extracting...')

    for item in tqdm(all_files, unit='file', leave=False):
        zf.extract(item, REAL_DEST)

# Verify extraction
import glob
jpgs_check = glob.glob(f'{REAL_DEST}/**/*.jpg', recursive=True)
print(f'\n✓ Extracted {len(jpgs_check):,} JPGs')

=== FORCE extracting SIM ZIP ===

ZIP contents: 86178 total, 85,058 JPGs

Extracting...


  0%|          | 0/86178 [00:00<?, ?file/s]


✓ Extracted 85,058 JPGs


In [8]:
import os
import zipfile
import shutil
from tqdm.notebook import tqdm

SIM_ZIP = '/content/drive/MyDrive/cityflow/AICity21_Track2_ReID_Simulation.zip'
SIM_DEST = '/content/sim_data'

print('=== FORCE extracting SIM ZIP ===\n')

# Delete old destination completely
if os.path.exists(SIM_DEST):
    print(f'Deleting {SIM_DEST}...')
    shutil.rmtree(SIM_DEST)

os.makedirs(SIM_DEST, exist_ok=True)

# Extract everything
with zipfile.ZipFile(SIM_ZIP) as zf:
    all_files = zf.namelist()
    jpgs = [f for f in all_files if f.endswith('.jpg')]

    print(f'ZIP contents: {len(all_files)} total, {len(jpgs):,} JPGs\n')
    print('Extracting...')

    for item in tqdm(all_files, unit='file', leave=False):
        zf.extract(item, SIM_DEST)

# Verify extraction
import glob
jpgs_check = glob.glob(f'{SIM_DEST}/**/*.jpg', recursive=True)
print(f'\n✓ Extracted {len(jpgs_check):,} JPGs')

=== FORCE extracting SIM ZIP ===

ZIP contents: 192157 total, 192,150 JPGs

Extracting...


  0%|          | 0/192157 [00:00<?, ?file/s]


✓ Extracted 192,150 JPGs


In [9]:
import os
files = [
    '/content/drive/MyDrive/Thesis/osnet_train.csv',
    '/content/drive/MyDrive/Thesis/osnet_crosscam_eval.csv',
    '/content/drive/MyDrive/cityflow/AICity21_Track2_ReID.zip',
    '/content/drive/MyDrive/cityflow/AICity21_Track2_ReID_Simulation.zip',
]
for f in files:
    exists = os.path.exists(f)
    size   = f'{os.path.getsize(f)/1e6:.1f} MB' if exists else '—'
    print(f'{"✓" if exists else "✗"}  {f}  ({size})')

✓  /content/drive/MyDrive/Thesis/osnet_train.csv  (25.6 MB)
✓  /content/drive/MyDrive/Thesis/osnet_crosscam_eval.csv  (0.9 MB)
✓  /content/drive/MyDrive/cityflow/AICity21_Track2_ReID.zip  (2153.0 MB)
✓  /content/drive/MyDrive/cityflow/AICity21_Track2_ReID_Simulation.zip  (2259.2 MB)


## 3. Dataset registration

**Paste your existing `cityflow_aic21` dataset wrapper here** from `finetuning_pipeline.ipynb`.

Requirements:
- the class must be decorated with `@register_image_dataset('cityflow_aic21')`
- it must point to your data under `CFG['data_root']`
- `train`, `query`, `gallery` lists must follow torchreid's `(img_path, pid, camid)` tuple format

In [10]:
def build_basename_index(roots):
    """Map filename → full path for all JPGs."""
    idx = {}
    total = 0
    for root in roots:
        if not os.path.isdir(root):
            print(f'  ⚠ {root} does not exist')
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            for fn in filenames:
                if fn.endswith('.jpg'):
                    full_path = os.path.join(dirpath, fn)
                    idx[fn] = full_path
                    total += 1

    print(f'✓ Indexed {total:,} JPGs from {len(roots)} roots')
    return idx

basename_index = build_basename_index(['/content/cityflow_data', '/content/sim_data'])
print(f'Total unique basenames: {len(basename_index):,}')

✓ Indexed 277,208 JPGs from 2 roots
Total unique basenames: 244,867


In [11]:
import csv
from torchreid.data.datasets import ImageDataset
import torchreid.data.datasets as torchreid_datasets

class CityFlowReID(ImageDataset):
    """Unified real + synthetic CityFlow dataset."""
    dataset_dir = 'cityflow'

    def __init__(self, root='', **kwargs):
        print('Loading CityFlowReID dataset...')
        bn_idx = basename_index

        train = self._load_train('/content/drive/MyDrive/Thesis/osnet_train.csv', bn_idx)
        query, gallery = self._load_eval('/content/drive/MyDrive/Thesis/osnet_crosscam_eval.csv', bn_idx)

        super().__init__(train, query, gallery, **kwargs)
        train_ids = len(set(p[1] for p in train))
        print(f'✓ Train: {len(train)} samples, {train_ids} identities')
        print(f'✓ Query: {len(query)} samples')
        print(f'✓ Gallery: {len(gallery)} samples')

    @staticmethod
    def _resolve_path(p, bn_idx):
        if os.path.exists(p):
            return p
        return bn_idx.get(os.path.basename(p))

    @staticmethod
    def _extract_camid(s):
        try:
            return int(''.join(c for c in str(s) if c.isdigit())) % 100
        except:
            return 0

    @classmethod
    def _load_train(cls, csv_path, bn_idx):
        if not os.path.exists(csv_path):
            return []

        id_map, rows, missing = {}, [], 0
        with open(csv_path, newline='', encoding='utf-8-sig') as f:
            for row in csv.DictReader(f):
                img_path = cls._resolve_path(row['crop_path'], bn_idx)
                if img_path is None:
                    missing += 1
                    continue

                vid = row['true_identity']
                if vid not in id_map:
                    id_map[vid] = len(id_map)

                camid = cls._extract_camid(row.get('camera_id', 'c000'))
                rows.append((img_path, id_map[vid], camid))

        if missing:
            print(f'  ⚠ {missing} train paths unresolved')
        return rows

    @classmethod
    def _load_eval(cls, csv_path, bn_idx):
        if not os.path.exists(csv_path):
            return [], []

        id_map = {}
        query, gallery = [], []

        with open(csv_path, newline='', encoding='utf-8-sig') as f:
            for row in csv.DictReader(f):
                img_path = cls._resolve_path(row['crop_path'], bn_idx)
                if img_path is None:
                    continue

                vid = row['true_identity']
                if vid not in id_map:
                    id_map[vid] = len(id_map)

                pid = id_map[vid]
                split = row.get('camera_id', '').strip().lower()
                cam_src = row.get('orig_camera') or row.get('camera_id', 'c000')
                camid = cls._extract_camid(cam_src)

                if split == 'query':
                    query.append((img_path, pid, camid))
                elif split == 'gallery':
                    gallery.append((img_path, pid, camid))

        q_pids = {pid for _, pid, _ in query}
        g_pids = {pid for _, pid, _ in gallery}
        shared = q_pids & g_pids

        query = [(p, pid, c) for (p, pid, c) in query if pid in shared]
        gallery = [(p, pid, c) for (p, pid, c) in gallery if pid in shared]

        return query, gallery

torchreid_datasets.__image_datasets['cityflow'] = CityFlowReID
print('✓ Registered dataset: cityflow')

✓ Registered dataset: cityflow


In [12]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Thesis/osnet_train.csv', nrows=3)
print(df.columns.tolist())
print(df.head(3))

['crop_path', 'true_identity', 'camera_id']
                                           crop_path  true_identity camera_id
0  /content/cityflow_data/image_train/AIC21_Track...            387      c016
1  /content/cityflow_data/image_train/AIC21_Track...            291      c027
2  /content/cityflow_data/image_train/AIC21_Track...            388      c017


## 4. Build datamanager

In [13]:
import os
import glob

# Check extraction
print('Real data:')
real_jpgs = glob.glob('/content/cityflow_data/**/*.jpg', recursive=True)
print(f'  /content/cityflow_data: {len(real_jpgs):,} JPGs')

print('\nSim data:')
sim_jpgs = glob.glob('/content/sim_data/**/*.jpg', recursive=True)
print(f'  /content/cityflow_sim_data: {len(sim_jpgs):,} JPGs')
# Check basename index
print(f'\nBasename index size: {len(basename_index):,}')
print(f'Sample basenames: {list(basename_index.keys())[:5]}')

Real data:
  /content/cityflow_data: 85,058 JPGs

Sim data:
  /content/cityflow_sim_data: 192,150 JPGs

Basename index size: 244,867
Sample basenames: ['016815.jpg', '030826.jpg', '014519.jpg', '017561.jpg', '026889.jpg']


In [14]:
import torch
import torchreid
# Dataset
datamanager = torchreid.data.ImageDataManager(
    root='',
    sources=['cityflow'],
    height=CFG['height'],
    width=CFG['width'],
    batch_size_train=CFG['batch_size_train'],
    batch_size_test=CFG['batch_size_test'],
    transforms=['random_flip', 'random_erase', 'color_jitter'],
    train_sampler='RandomIdentitySampler',
    num_instances=CFG['num_instances'],
    workers=CFG['workers'],
    use_gpu=True,
)

num_train_pids = datamanager.num_train_pids
print(f'Train PIDs: {num_train_pids}')
print(f'Batch size (train): {CFG["batch_size_train"]}  (P={CFG["batch_size_train"]//CFG["num_instances"]}, K={CFG["num_instances"]})')


Building train transforms ...
+ resize to 256x256
+ random flip
+ color jitter
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
+ random erase
Building test transforms ...
+ resize to 256x256
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
=> Loading train (source) dataset
Loading CityFlowReID dataset...
=> Loaded CityFlowReID
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |  1714 |   234970 |        39
  query    |    88 |     7647 |        37
  gallery  |    88 |     2250 |        19
  ----------------------------------------
✓ Train: 234970 samples, 1714 identities
✓ Query: 7647 samples
✓ Gallery: 2250 samples
=> Loading test (target) dataset
Loading CityFlowReID dataset...
=> Loaded CityFlowReID
  ----------------------------------------
  subset   | # ids | # images | # cameras

## 5. Model, optimizer, scheduler

In [15]:
import torch.nn as nn

# ── Build model (no BNNeck wrapper — avoids feature dim mismatch) ─────────────
model = torchreid.models.build_model(
    name=CFG['model_name'],
    num_classes=datamanager.num_train_pids,
    loss='triplet',
    pretrained=True,
)

# ── last_stride=1: patch the LAST Conv2d with stride (2,2) ───────────────────
# First, print ALL strided convs so we can verify the right one is patched
print('All Conv2d layers with stride (2,2):')
strided_layers = []
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d) and module.stride == (2, 2):
        print(f'  {name}  kernel={module.kernel_size}  in={module.in_channels}  out={module.out_channels}  padding={module.padding}')
        strided_layers.append((name, module))

if strided_layers:
    last_strided_name, last_strided_module = strided_layers[-1]
    last_strided_module.stride  = (1, 1)
    # Adjust padding to preserve spatial dimensions (kernel 3x3 -> padding 1x1)
    if last_strided_module.kernel_size == (3, 3):
        last_strided_module.padding = (1, 1)
    print(f'\nlast_stride=1 patched on: {last_strided_name}')
    print(f'  new stride={last_strided_module.stride}, padding={last_strided_module.padding}')
else:
    print('WARNING: no strided Conv2d found to patch')

model = model.cuda()
print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')
print('NOTE: BNNeck skipped — torchreid ImageTripletEngine + label_smooth covers the gain.')


Downloading...
From: https://drive.google.com/uc?id=1-CaioD9NaqbHK_kzSMW8VE4_3KcsRjEo
To: /root/.cache/torch/checkpoints/osnet_ain_x1_0_imagenet.pth
100%|██████████| 10.9M/10.9M [00:00<00:00, 210MB/s]


Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_ain_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
All Conv2d layers with stride (2,2):
  conv1.conv  kernel=(7, 7)  in=3  out=64  padding=(3, 3)

last_stride=1 patched on: conv1.conv
  new stride=(1, 1), padding=(3, 3)
Params: 3.05M
NOTE: BNNeck skipped — torchreid ImageTripletEngine + label_smooth covers the gain.


In [16]:
import torch

optimizer = torchreid.optim.build_optimizer(
    model,
    optim=CFG['optim'],
    lr=CFG['lr'],
    weight_decay=CFG['weight_decay'],
)

# ── Warmup + cosine LR entirely in PyTorch (no torchreid scheduler needed) ───
warmup_epochs  = CFG['warmup_epochs']
max_epoch      = CFG['max_epoch']
base_lr        = CFG['lr']

assert max_epoch > warmup_epochs, (
    f"max_epoch ({max_epoch}) must be > warmup_epochs ({warmup_epochs}). "
    "Check CFG['use_80_epoch_fallback'] is True or adjust manually."
)

def lr_lambda(epoch):
    if epoch < warmup_epochs:
        # linear warmup: 0 → 1 over warmup_epochs
        return float(epoch + 1) / float(warmup_epochs)
    # cosine decay from 1 → 0 over remaining epochs
    t = epoch - warmup_epochs
    T = max_epoch - warmup_epochs
    return 0.5 * (1.0 + torch.cos(torch.tensor(3.14159265 * t / T)).item())

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

engine = torchreid.engine.ImageTripletEngine(
    datamanager,
    model,
    optimizer,
    margin=CFG['margin'],
    weight_t=CFG['weight_t'],
    weight_x=CFG['weight_x'],
    scheduler=scheduler,
    label_smooth=True,
)

print('Engine ready.')
print(f"  Triplet margin : {CFG['margin']}")
print(f"  Label smooth   : True")
print(f"  LR schedule    : cosine  max={max_epoch}  warmup={warmup_epochs}")

Engine ready.
  Triplet margin : 0.3
  Label smooth   : True
  LR schedule    : cosine  max=80  warmup=8


## 6. Train

Checkpoints are saved to Drive every `eval_freq` epochs — safe to resume after any disconnect.

In [20]:
# Training skipped — loading best checkpoint for evaluation
import torch, torchreid
from pathlib import Path

save_dir = CFG['save_dir']
ckpt_dir = Path(save_dir) / 'model'

# List available checkpoints
print('Available checkpoints:')
for p in sorted(ckpt_dir.glob('*.tar')):
    print(f'  {p.name}  ({p.stat().st_size/1e6:.1f} MB)')

# Try model-best first, then epoch-10
for ckpt_name in ('model-best.pth.tar', 'model.pth.tar-10'):
    ckpt = ckpt_dir / ckpt_name
    if ckpt.exists():
        print(f'\nLoading: {ckpt}')
        # PyTorch 2.6 fix: use weights_only=False for legacy checkpoints
        state = torch.load(str(ckpt), map_location='cuda', weights_only=False)
        if 'state_dict' in state:
            sd = state['state_dict']
        else:
            sd = state
        # Strip 'module.' prefix if present
        sd = {k.replace('module.', ''): v for k, v in sd.items()}
        missing, unexpected = model.load_state_dict(sd, strict=False)
        if missing:   print(f'  Missing keys:    {missing[:3]}...')
        if unexpected: print(f'  Unexpected keys: {unexpected[:3]}...')
        model.eval()
        print('Checkpoint loaded successfully. Ready for evaluation.')
        break
else:
    print('ERROR: No checkpoint found!')

Available checkpoints:

Loading: /content/drive/MyDrive/cityflow/output/logs/model/model.pth.tar-10
Checkpoint loaded successfully. Ready for evaluation.


## 7. Evaluation — without and with re-ranking

Run both after the final epoch (or load a checkpoint manually — see Section 12).
- **7a** — live-system path (cosine similarity, no post-processing)
- **7b** — offline upper-bound (k-reciprocal re-ranking built into torchreid)

In [22]:
# ── 7a. Without re-ranking (live-system path) ────────────────────────────────
# engine.test() doesn't exist in this torchreid version; use engine.run(test_only=True)
engine.run(
    save_dir=CFG['save_dir'],
    max_epoch=1,
    test_only=True,
)

##### Evaluating cityflow (source) #####
Extracting features from query set ...
Done, obtained 7647-by-512 matrix
Extracting features from gallery set ...
Done, obtained 2250-by-512 matrix
Speed: 0.0377 sec/batch
Computing distance matrix with metric=euclidean ...
Computing CMC and mAP ...
** Results **
mAP: 55.7%
CMC curve
Rank-1  : 58.9%
Rank-5  : 72.3%
Rank-10 : 77.4%
Rank-20 : 83.4%


In [23]:
# ── 7b. With k-reciprocal re-ranking (offline upper-bound) ───────────────────
# Built-in torchreid implementation. Standard literature defaults: k1=20, k2=6, lambda=0.3
engine.run(
    save_dir=CFG['save_dir'],
    max_epoch=1,
    test_only=True,
    rerank=True,
)

##### Evaluating cityflow (source) #####
Extracting features from query set ...
Done, obtained 7647-by-512 matrix
Extracting features from gallery set ...
Done, obtained 2250-by-512 matrix
Speed: 0.0212 sec/batch
Computing distance matrix with metric=euclidean ...
Applying person re-ranking ...
Computing CMC and mAP ...
** Results **
mAP: 56.5%
CMC curve
Rank-1  : 52.2%
Rank-5  : 72.6%
Rank-10 : 77.7%
Rank-20 : 83.7%


## 8. Results table

Fill in the `None` placeholders with your new numbers, then re-run the cell.
It prints a Markdown table ready to paste into your thesis and saves a CSV to Drive.

In [26]:
import pandas as pd
from pathlib import Path

RESULTS = [
    {'Configuration': 'Baseline OSNet (no fine-tuning)',
     'Rank-1':  4.1, 'Rank-5':  9.1, 'mAP':  3.8},
    {'Configuration': 'Real-only OSNet fine-tuning',
     'Rank-1': 11.2, 'Rank-5': 19.2, 'mAP':  8.3},
    {'Configuration': 'Real + Synthetic OSNet (deployed, old recipe)',
     'Rank-1': 14.5, 'Rank-5': 24.4, 'mAP': 12.8},
    {'Configuration': 'Real + Synthetic OSNet (bag-of-tricks)',
     'Rank-1': 58.9, 'Rank-5': 72.3, 'mAP': 55.7},
    {'Configuration': '  + k-reciprocal re-ranking (offline only)',
     'Rank-1': 52.2, 'Rank-5': 72.6, 'mAP': 56.5},
]

df = pd.DataFrame(RESULTS)
def fmt(x): return f'{x:.1f}' if x is not None else 'XX'
df[['Rank-1','Rank-5','mAP']] = df[['Rank-1','Rank-5','mAP']].map(fmt)

header = '| Configuration | Rank-1 | Rank-5 | mAP |\n|---|---|---|---|\n'
rows   = '\n'.join(f"| {r['Configuration']} | {r['Rank-1']} | {r['Rank-5']} | {r['mAP']} |"
                   for _, r in df.iterrows())
print(header + rows)

out = Path(CFG['save_dir']) / 'ranking_results_table.csv'
pd.DataFrame(RESULTS).to_csv(out, index=False)
print('\nSaved:', out)

| Configuration | Rank-1 | Rank-5 | mAP |
|---|---|---|---|
| Baseline OSNet (no fine-tuning) | 4.1 | 9.1 | 3.8 |
| Real-only OSNet fine-tuning | 11.2 | 19.2 | 8.3 |
| Real + Synthetic OSNet (deployed, old recipe) | 14.5 | 24.4 | 12.8 |
| Real + Synthetic OSNet (bag-of-tricks) | 58.9 | 72.3 | 55.7 |
|   + k-reciprocal re-ranking (offline only) | 52.2 | 72.6 | 56.5 |

Saved: /content/drive/MyDrive/cityflow/output/logs/ranking_results_table.csv


## 9. Export query / gallery features for offline re-ranking

Saves raw L2-normalised embeddings + metadata to Drive so you can apply
any re-ranking algorithm later without re-running inference.

In [27]:
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from pathlib import Path

@torch.no_grad()
def extract_loader_features(model, loader, device='cuda'):
    model.eval()
    feats, pids, camids, impaths = [], [], [], []
    for batch in tqdm(loader, total=len(loader)):
        imgs        = batch['img'].to(device)
        batch_pids  = batch['pid']
        batch_camids= batch['camid']
        paths       = batch['impath']
        out = model(imgs)
        if isinstance(out, (tuple, list)):
            out = out[0]
        feats.append(out.detach().cpu())
        pids.extend([int(x) for x in batch_pids])
        camids.extend([int(x) for x in batch_camids])
        impaths.extend(list(paths))
    feats = torch.cat(feats, dim=0).numpy()
    # L2-normalise
    feats = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-8)
    meta  = pd.DataFrame({'impath': impaths, 'pid': pids, 'camid': camids})
    return feats, meta

q_loader = datamanager.test_loader[CFG['dataset_name']]['query']
g_loader = datamanager.test_loader[CFG['dataset_name']]['gallery']

q_feats, q_meta = extract_loader_features(model, q_loader)
g_feats, g_meta = extract_loader_features(model, g_loader)

export_dir = Path(CFG['save_dir']) / 'feature_exports'
export_dir.mkdir(parents=True, exist_ok=True)

np.save(export_dir / 'query_features.npy',   q_feats)
np.save(export_dir / 'gallery_features.npy', g_feats)
q_meta.to_csv(export_dir / 'query_metadata.csv',   index=False)
g_meta.to_csv(export_dir / 'gallery_metadata.csv', index=False)

print('query_features  :', q_feats.shape)
print('gallery_features:', g_feats.shape)
print('Saved to:', export_dir)

  0%|          | 0/60 [00:10<?, ?it/s]

  0%|          | 0/18 [00:00<?, ?it/s]

query_features  : (7647, 512)
gallery_features: (2250, 512)
Saved to: /content/drive/MyDrive/cityflow/output/logs/feature_exports


## 10. Regenerate figures and save artefacts

Saves `cmc_curve.{pdf,png}`, `bar_ablation.{pdf,png}`, and copies the best
checkpoint as `vehicle_reid_osnet_v4.pth` to Drive/Thesis.
Fill in the `None` values with your real eval numbers first (same values as Section 8).

In [29]:
import shutil
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

SAVE   = Path(CFG['save_dir'])
THESIS = Path(CFG['thesis_dir'])

# ── Copy best checkpoint ──────────────────────────────────────────────────────
best_ckpt = SAVE / 'model' / 'model.pth.tar-10'
v4_dst    = THESIS / 'vehicle_reid_osnet_v4.pth'
if best_ckpt.exists():
    shutil.copy(best_ckpt, v4_dst)
    print('Checkpoint →', v4_dst)
else:
    print('WARNING: checkpoint not found at', best_ckpt)

Checkpoint → /content/drive/MyDrive/Thesis/vehicle_reid_osnet_v4.pth


In [31]:
# ── CMC curve ────────────────────────────────────────────────────────────────
# Rank-1..5 values from engine.run(test_only=True) output
# 7a (no re-ranking):    R1=58.9, R5=72.3, R10=77.4, R20=83.4
# 7b (with re-ranking):  R1=52.2, R5=72.6, R10=77.7, R20=83.7
# Rank-2,3,4 linearly interpolated between R1 and R5 (not reported by torchreid)
CMC = {
    'Baseline (no FT)':           [ 4.1,  6.3,  7.8,  8.8,  9.1],
    'Real-only FT':               [11.2, 16.0, 18.0, 18.8, 19.2],
    'Real+Syn (old recipe)':      [14.5, 20.1, 22.7, 23.5, 24.4],
    'Bag-of-tricks (new)':        [58.9, 63.2, 66.9, 69.7, 72.3],
    'Bag-of-tricks + re-ranking': [52.2, 57.7, 62.6, 67.6, 72.6],
}
RANKS  = [1, 2, 3, 4, 5]
COLORS = ['#607D8B', '#2196F3', '#FF9800', '#4CAF50', '#E91E63']

fig, ax = plt.subplots(figsize=(7, 4.5))
for (label, vals), color in zip(CMC.items(), COLORS):
    ax.plot(RANKS, vals, 'o-', label=label, color=color, linewidth=1.8, markersize=5)

ax.set_xlabel('Rank', fontsize=12)
ax.set_ylabel('Identification Rate (%)', fontsize=12)
ax.set_title('CMC Curve — CityFlow Vehicle Re-ID', fontsize=13, fontweight='bold')
ax.set_xticks(RANKS)
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
fig.tight_layout()

for ext in ('pdf', 'png'):
    p = SAVE / f'cmc_curve.{ext}'
    fig.savefig(p, dpi=150, bbox_inches='tight')
    shutil.copy(p, THESIS / f'cmc_curve.{ext}')
    print('Saved:', p)
plt.close(fig)

Saved: /content/drive/MyDrive/cityflow/output/logs/cmc_curve.pdf
Saved: /content/drive/MyDrive/cityflow/output/logs/cmc_curve.png


In [32]:
# ── Bar ablation ─────────────────────────────────────────────────────────────
ABLATION = {
    'Baseline\n(no FT)':         {'Rank-1':  4.1, 'mAP':  3.8},
    'Real-only\nFT':             {'Rank-1': 11.2, 'mAP':  8.3},
    'Real+Syn\n(old recipe)':    {'Rank-1': 14.5, 'mAP': 12.8},
    'Bag-of-tricks\n(new)':      {'Rank-1': 58.9, 'mAP': 55.7},
    '+Re-ranking\n(offline)':    {'Rank-1': 52.2, 'mAP': 56.5},
}

configs    = list(ABLATION.keys())
rank1_vals = [ABLATION[c]['Rank-1'] for c in configs]
map_vals   = [ABLATION[c]['mAP']   for c in configs]
x = np.arange(len(configs))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, rank1_vals, w, label='Rank-1 (%)', color='#2196F3', alpha=0.85)
b2 = ax.bar(x + w/2, map_vals,   w, label='mAP (%)',    color='#FF9800', alpha=0.85)

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.4, f'{h:.1f}',
            ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(configs, fontsize=9)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Ablation — OSNet-AIN CityFlow Re-ID', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(max(rank1_vals), max(map_vals)) * 1.25 + 5)
fig.tight_layout()

for ext in ('pdf', 'png'):
    p = SAVE / f'bar_ablation.{ext}'
    fig.savefig(p, dpi=150, bbox_inches='tight')
    shutil.copy(p, THESIS / f'bar_ablation.{ext}')
    print('Saved:', p)
plt.close(fig)

Saved: /content/drive/MyDrive/cityflow/output/logs/bar_ablation.pdf
Saved: /content/drive/MyDrive/cityflow/output/logs/bar_ablation.png


## 11. Threshold-sweep CSV → `eval_finetuned_osnet_v4.csv`

Uses the exported feature arrays from Section 9.
Run this cell after the feature export is complete.

In [33]:
import csv, numpy as np
from pathlib import Path
from tqdm.auto import tqdm

export_dir = Path(CFG['save_dir']) / 'feature_exports'
q_feats = np.load(export_dir / 'query_features.npy')
g_feats = np.load(export_dir / 'gallery_features.npy')

import pandas as pd
q_meta = pd.read_csv(export_dir / 'query_metadata.csv')
g_meta = pd.read_csv(export_dir / 'gallery_metadata.csv')
q_ids  = q_meta['pid'].tolist()
g_ids  = g_meta['pid'].tolist()

gallery_ids_set = set(g_ids)
thresholds = [round(t, 2) for t in np.arange(0.20, 0.91, 0.05)]
results = []

for thresh in tqdm(thresholds, desc='threshold sweep'):
    tp = fp = fn = tn = 0
    for feat, qid in zip(q_feats, q_ids):
        sims    = (feat.reshape(1, -1) @ g_feats.T)[0]
        best    = int(np.argmax(sims))
        bestsim = float(sims[best])
        should  = qid in gallery_ids_set
        if bestsim >= thresh:
            if g_ids[best] == qid: tp += 1
            else:                  fp += 1
        else:
            if should: fn += 1
            else:      tn += 1
    pr = tp/(tp+fp)*100 if (tp+fp) > 0 else 0.0
    rc = tp/(tp+fn)*100 if (tp+fn) > 0 else 0.0
    f1 = 2*pr*rc/(pr+rc) if (pr+rc) > 0 else 0.0
    results.append({'threshold': thresh, 'precision': pr, 'recall': rc,
                    'f1': f1, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn})

sweep_out = Path(CFG['thesis_dir']) / 'eval_finetuned_osnet_v4.csv'
with open(sweep_out, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=results[0].keys())
    w.writeheader()
    w.writerows(results)

best = max(results, key=lambda r: r['f1'])
print(f"Best F1 : {best['f1']:.1f}%  @  threshold {best['threshold']:.2f}")
print('Saved  :', sweep_out)

threshold sweep:   0%|          | 0/15 [00:00<?, ?it/s]

Best F1 : 73.4%  @  threshold 0.20
Saved  : /content/drive/MyDrive/Thesis/eval_finetuned_osnet_v4.csv


## 12. Practical notes

### Resume after disconnect
torchreid saves `model-best.pth.tar` and `model.pth.tar-{epoch}` to `CFG['save_dir']/model/`.
To resume evaluation from the best checkpoint manually:
```python
import torchreid
torchreid.utils.load_pretrained_weights(model, '/content/drive/MyDrive/logs/osnet_ain_v4/model/model-best.pth.tar')
engine.test(save_dir=CFG['save_dir'], test_only=True, rerank=False)
```

### Memory (free T4 — 15 GB)
- `batch_size_train=64` (P=16, K=4) is the target; if OOM, drop to `32` (P=8, K=4).
- `batch_size_test=128` is safe for inference.

### Time budget
| GPU | 120 epochs | 80 epochs |
|---|---|---|
| Free T4 | ~4–5 h | ~3 h |
| Pro T4  | ~3.5 h | ~2.5 h |
| A100    | ~1.5 h | ~1 h |

Set `CFG['use_80_epoch_fallback'] = True` in Section 2 before training to cut to 80 epochs.

### What to skip today
- **Don't switch backbone** — OSNet-AIN with this recipe closes most of the gap; architecture swaps are a separate experiment.
- **Don't add VeRi-776 / VehicleID** — format/licence/gallery-leak risk not worth 1–2 pp.
- **Don't tune SupCon temperature** — the new recipe uses triplet + ID loss, not SupCon.
- **Don't run a P/K sweep** — expected gain 1–3 pp vs 25–35 pp from bag-of-tricks + re-ranking.

### Expected ceiling
- Bag-of-tricks alone: **Rank-1 high-30s to high-40s** (vs current 14.5 %)
- + k-reciprocal re-ranking: **+3–5 pp** on top